# Sparse-grid campaign dry run — all four baselines

Drives `ScanStudy` through the full campaign loop for **129015, 129038, 132543,
132588** and stops at the cheaseBS call. Everything upstream of the equilibrium
solve is the production path: the nominal mtanh fits, the axis transforms, the
per-point GENE profiles and iterdb, the sparse-grid sampler, EQDSK retagging,
and the GENE parameters writer.

**What this is for.** Four discharges through one code path, so a difference
between them is a property of the discharge rather than of a notebook copy.
That is the failure this replaces: the pilot notebooks diverged, and fixes
applied to one silently missed the others.

**What it cannot tell you.** No CHEASE binary, so no equilibrium is
reconstructed. `allow_cheasebs_fallback=True` substitutes the *source* gfile for
each point so the grid can advance — the profiles are real, the equilibria are
not. Nothing here says anything about whether a point's equilibrium is
physical; that needs NERSC.

**The profiles are the deliverable.** Per-point `profiles_e/i/z` written out and
compared is the cheapest evidence that a transform actually took, and is worth
keeping as a routine campaign check rather than only a dry-run one — a scan
whose profiles are identical between points runs to completion and returns
growth rates that mean nothing.

## Inputs

Set `SG_LIB_PATH` and `BASE_PARAMETERS` for the machine you are on. Everything
else is per-discharge below.

In [ ]:
import os, sys, json, warnings
import numpy as np

from TPED.projects.GENE_pipelines.src.scan_study import StudyConfig, ScanStudy

# sg_lib clone. None reads the SG_LIB_PATH config key — correct on a configured
# machine, so leave it None on NERSC.
SG_LIB_PATH = r"C:/Users/joesc/git/sensitivity-driven-sparse-grid-approx"

# Baseline GENE parameters template. STAND-IN: this is 129015's, used for all
# four because only the parameters-writing mechanics are under test here, not
# the physics content of the namelist. A real campaign needs the template that
# belongs to its own discharge and radius.
BASE_PARAMETERS = (r"C:/Users/joesc/git/TPED/data/discharges/NSTX129015/"
                   r"r_0.85_NE/scanfiles0000/parameters")

DISCHARGE_ROOT = r"C:/Users/joesc/git/ST_research/NSTXU_discharges"
WORKROOT = "runs"

# Scale factors on each discharge's own nominal mtanh fit. NOT pedestal-top
# fractions: scale_height multiplies the mtanh step amplitude, and the
# conversion differs per discharge and variable. fit_report() prints the
# measured slope for each, which is the number that converts these into the
# survey's absolute bounds.
BOUNDS = {"Te_ped_scale": (0.7, 1.3), "ne_ped_scale": (0.7, 1.3)}

TARGET_POINTS = 5

# Radii the acceptance gate scores q at — per discharge, since they are where
# GENE will actually be run. 132543/132588 are the q=4 and q=5 surfaces used by
# the existing linear scans; 129015 is its r_0.85 case. 129038 has no scan of
# record here, so its entry is a PLACEHOLDER and its gate verdict should not be
# read as meaningful.
#
# `extra` goes straight into DischargeData. 129038 needs it: its directory holds
# five pfiles (p129038.00400 plus _original, _odd_rotation_profile,
# _rotation_from_420, and .00420), so auto-discovery refuses to guess and
# raises. That refusal is correct -- silently picking one would put an
# unannounced rotation variant under an entire scan -- and it is exactly why
# the study takes DischargeData kwargs rather than a bare directory.
DISCHARGES = [
    {"shot": 129015, "radii": (0.85,),        "placeholder_radii": False, "extra": {}},
    {"shot": 129038, "radii": (0.85,),        "placeholder_radii": True,
     "extra": {"pfile": os.path.join(DISCHARGE_ROOT, "129038", "p129038.00400")}},
    {"shot": 132543, "radii": (0.736, 0.825), "placeholder_radii": False, "extra": {}},
    {"shot": 132588, "radii": (0.736, 0.825), "placeholder_radii": False, "extra": {}},
]

for d in DISCHARGES:
    d["dirpath"] = os.path.join(DISCHARGE_ROOT, str(d["shot"]))
    notes = []
    if d["placeholder_radii"]:
        notes.append("placeholder radii")
    if d["extra"]:
        notes.append("explicit " + ",".join(d["extra"]))
    print(f"{d['shot']}  {'ok' if os.path.isdir(d['dirpath']) else 'NOT FOUND'}"
          f"  {d['dirpath']}" + (f"   ({'; '.join(notes)})" if notes else ""))

## Run each discharge through the same code path

In [ ]:
studies, failures = {}, {}

for d in DISCHARGES:
    shot = d["shot"]
    print("\n" + "=" * 74)
    print(f"  {shot}")
    print("=" * 74)
    try:
        cfg = StudyConfig(
            shot=shot,
            discharge_kwargs={"input_dir": d["dirpath"], **d["extra"]},
            base_parameters=BASE_PARAMETERS,
            bounds=BOUNDS,
            analysis_radii=d["radii"],
            target_points=TARGET_POINTS,
            workroot=os.path.join(WORKROOT, str(shot)),
            sg_lib_path=SG_LIB_PATH,
            allow_cheasebs_fallback=True,   # no CHEASE locally; see the header
        )
        study = ScanStudy(cfg)
        study.load_seed()
        d["fit"] = study.fit_report()
        study.run()
        studies[shot] = study
    except Exception as exc:
        # A discharge that cannot even be set up is the finding, not a reason to
        # abandon the other three.
        failures[shot] = f"{type(exc).__name__}: {exc}"
        print(f"\n!! {shot} FAILED: {failures[shot]}")

print(f"\n{len(studies)}/{len(DISCHARGES)} discharges completed")
for shot, err in failures.items():
    print(f"  {shot}: {err}")

## Cross-discharge comparison

The columns that matter differ in kind. `rms_rel` says whether the mtanh fit is
good enough for the axes to mean anything. `slope` is the scale-factor to
pedestal-top conversion, which the Phase-0 bounds freeze needs per discharge.
`unity %` is how far the unity transform moves the pedestal top *before* any
axis is applied — a bias every point in that discharge's scan inherits.

In [ ]:
rows = []
for d in DISCHARGES:
    shot = d["shot"]
    fit = d.get("fit") or {}
    st = studies.get(shot)
    for var, f in sorted(fit.items()):
        rows.append({
            "shot": shot, "var": var,
            "rms_rel": f["rms_relative"],
            "ped_top": f["ped_top"],
            "slope": f.get("ped_top_slope", float("nan")),
            "unity_pct": 100 * f.get("unity_offset", float("nan")),
            "points": len(st.campaign.ledger.entries) if st else 0,
        })

hdr = (f"{'shot':>7} {'var':<4} {'rms_rel':>9} {'ped_top':>11} {'slope':>7} "
       f"{'unity %':>8} {'points':>7}")
print(hdr); print("-" * len(hdr))
for r in rows:
    print(f"{r['shot']:>7} {r['var']:<4} {r['rms_rel']:>9.4f} "
          f"{r['ped_top']:>11.4g} {r['slope']:>7.4f} {r['unity_pct']:>8.2f} "
          f"{r['points']:>7}")

print("\nbounds in pedestal-top terms, per discharge:")
for r in rows:
    lo, hi = BOUNDS.get(f"{r['var']}_ped_scale", (float('nan'),) * 2)
    print(f"  {r['shot']} {r['var']:<3} scale [{lo}, {hi}]  ->  pedestal top "
          f"{(lo-1)*r['slope']*100:+.1f}% to {(hi-1)*r['slope']*100:+.1f}%")

## Did the transform take? — profiles per point

Reads each point's written `profiles_e` back off disk and compares. The
duplicate check is the one that matters: two points sharing a profile means the
transform silently did not apply, while every downstream stage still succeeds.

In [ ]:
def read_profiles(path):
    data = np.loadtxt(path, comments="#")
    return data[:, 0], data[:, 2], data[:, 3]     # rhot, T, n

profile_data = {}
for shot, st in studies.items():
    print(f"\n{shot}")
    entries = sorted(st.campaign.ledger.entries.values(), key=lambda e: e.point_id)
    curves, dupes = [], []
    for e in entries:
        p = os.path.join(e.savedir or "", "profiles_e")
        if not os.path.exists(p):
            print(f"  {e.point_id}  no profiles_e"); continue
        x, Te, ne = read_profiles(p)
        ped = (x >= 0.90) & (x <= 0.98)
        curves.append({"point_id": e.point_id, "tag": st.tag(e.point),
                       "x": x, "Te": Te, "ne": ne,
                       "Te_ped": float(np.max(Te[ped])),
                       "ne_ped": float(np.max(ne[ped]))})
    for i, a in enumerate(curves):
        for b in curves[i + 1:]:
            if (len(a["Te"]) == len(b["Te"])
                    and np.allclose(a["Te"], b["Te"], rtol=1e-12)
                    and np.allclose(a["ne"], b["ne"], rtol=1e-12)):
                dupes.append((a["point_id"], b["point_id"]))
    for c in curves:
        print(f"  {c['point_id']:<12} {c['tag']:<28} "
              f"Te_ped {c['Te_ped']:>9.4g}   ne_ped {c['ne_ped']:>10.4g}")
    if dupes:
        print(f"  !! {len(dupes)} identical profile pair(s): {dupes}")
    elif len(curves) > 1:
        print(f"  all {len(curves)} profiles differ")
    profile_data[shot] = curves

In [ ]:
import matplotlib.pyplot as plt

n = len(profile_data)
if n:
    fig, axes = plt.subplots(2, n, figsize=(4.2 * n, 7), squeeze=False)
    for j, (shot, curves) in enumerate(sorted(profile_data.items())):
        for c in curves:
            axes[0][j].plot(c["x"], c["Te"], lw=1.2, label=c["tag"])
            axes[1][j].plot(c["x"], c["ne"], lw=1.2)
        axes[0][j].set_title(str(shot)); axes[0][j].set_ylabel("Te (keV)")
        axes[1][j].set_ylabel("ne (1e19 m^-3)")
        for ax in (axes[0][j], axes[1][j]):
            ax.set_xlim(0.85, 1.0); ax.set_xlabel("rho_tor")
        axes[0][j].legend(fontsize=6)
    fig.suptitle("per-point profiles written for cheaseBS, by discharge")
    plt.tight_layout(); plt.show()

## What was written, and what is still missing

`parameters` files are the actual pipeline output. `verify_parameters` reads
each back and checks it says what the campaign asked: no scan axis leaked into
the namelist, `iterdb_file` and `geomfile` point at this point's own files.

The stand-in equilibria are reported separately and deliberately — they are the
one thing here that is not real.

In [ ]:
summary = {}
for shot, st in sorted(studies.items()):
    entries = list(st.campaign.ledger.entries.values())
    written = [e for e in entries if e.rundir]
    stand_in = [e for e in entries if (e.acceptance or {}).get("stand_in")]
    problems = st.verify_parameters()
    print(f"{shot}: {len(entries)} point(s), {len(written)} parameters file(s), "
          f"{len(stand_in)} stand-in equilibri{'um' if len(stand_in)==1 else 'a'}")
    print(f"        stop: {st.stop_reason}")
    for pid, msg in problems:
        print(f"        !! {pid}: {msg}")
    if not problems:
        print(f"        parameters verified clean")
    summary[shot] = {
        "workdir": st.workdir, "points": len(entries),
        "parameters_written": len(written), "stand_in": len(stand_in),
        "stop_reason": st.stop_reason,
        "problems": [list(p) for p in problems],
        "fit": {v: {k: val for k, val in f.items() if k != "fit_params"}
                for v, f in (next((d.get("fit") or {}) for d in DISCHARGES
                                  if d["shot"] == shot)).items()},
    }

out = os.path.join(WORKROOT, "four_discharge_dryrun.json")
os.makedirs(WORKROOT, exist_ok=True)
with open(out, "w") as f:
    json.dump({"summary": summary, "failures": failures,
               "bounds": BOUNDS, "target_points": TARGET_POINTS,
               "base_parameters": BASE_PARAMETERS,
               "equilibria_are_stand_ins": True}, f, indent=1, default=str)
print(f"\nwritten: {out}")
print("\nREMINDER: every equilibrium above is the source gfile, not a "
      "reconstruction. Re-run on NERSC with allow_cheasebs_fallback=False "
      "before reading anything into the gate verdicts.")